In [1]:

import time

# 显式声明需要 joblib（旧版 sklearn.externals.joblib 已弃用）
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
# 梯度提升库（注意 CatBoost 与 scikit-learn 的交互方式）
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
# Core Scikit-learn 1.4+ 兼容导入
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [2]:
# 1. 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")

# 2. 从原始数据提取标签（如果未保存 y.npy）
# 如果已保存 y.npy，直接加载：
y = np.load("y.npy")

# 3. 划分训练集与验证集
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


In [3]:


# Classifiers
classifiers = {
    "LogisticRegression": LogisticRegression(random_state=0),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=0, probability=True),
    "RandomForest": RandomForestClassifier(random_state=0),
    "LGBM": lgb.LGBMClassifier(random_state=0),
    "CatBoost": CatBoostClassifier(random_state=0, verbose=False),
    "NaiveBayes": GaussianNB()
}

# Grids for grid search
LR_grid = {
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'C': [0.5, 1, 1.5],
    'max_iter': [100]
}

KNN_grid = {'n_neighbors': [3, 5, 7],
            'p': [1, 2]}

SVC_grid = {'C': [0.5, 1, 1.5],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale']}

RF_grid = {'n_estimators': [50, 100, 150],
           'max_depth': [4, 8]}

boosted_grid = {'n_estimators': [50, 100],
                'learning_rate': [0.05, 0.1]}

CatBoost_grid = {'n_estimators': [50, 100],
                 'learning_rate': [0.05, 0.1],
                 'depth': [4, 6]}

NB_grid = {'var_smoothing': [1e-10, 1e-9, 1e-8]}

# Dictionary of all grids
grid = {
    "LogisticRegression": LR_grid,
    "KNN": KNN_grid,
    "SVC": SVC_grid,
    "RandomForest": RF_grid,
    "LGBM": boosted_grid,
    "CatBoost": CatBoost_grid,
    "NaiveBayes": NB_grid
}

i = 0
clf_best_params = classifiers.copy()
valid_scores = pd.DataFrame({'Classifier': classifiers.keys(), 'Validation Accuracy': np.zeros(len(classifiers)),
                             'Training Time': np.zeros(len(classifiers))})

for key, classifier in classifiers.items():
    start = time.time()
    clf = GridSearchCV(estimator=classifier, param_grid=grid[key], n_jobs=-1, cv=5)

    # Train and score
    clf.fit(X_train, y_train)
    valid_scores.iloc[i, 1] = clf.score(X_valid, y_valid)

    # Save trained model
    clf_best_params[key] = clf.best_params_

    # Print iteration and training time
    stop = time.time()
    valid_scores.iloc[i, 2] = np.round((stop - start) / 60, 2)

    print('Model:', key)
    print('Training time (mins):', valid_scores.iloc[i, 2])
    print('Best parameters:', clf_best_params[key])
    print('Validation accuracy:', valid_scores.iloc[i, 1])
    print('')
    i += 1

Model: LogisticRegression
Training time (mins): 0.02
Best parameters: {'C': 0.5, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Validation accuracy: 0.7705577918343876

Model: KNN
Training time (mins): 0.01
Best parameters: {'n_neighbors': 7, 'p': 1}
Validation accuracy: 0.7682576193214491

Model: SVC
Training time (mins): 0.35
Best parameters: {'C': 1.5, 'gamma': 'scale', 'kernel': 'rbf'}
Validation accuracy: 0.7935595169637722

Model: RandomForest
Training time (mins): 0.04
Best parameters: {'max_depth': 8, 'n_estimators': 150}
Validation accuracy: 0.7843588269120184

[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Number of positive: 2802, number of negative: 2762
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Number of positive: 2802, number of negative: 2762
[LightGBM] [Info] Number of positive: 2801, number of negative: 2762
[Li

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2801, number of negative: 2762
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.136927 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2439
[LightGBM] [Info] Number of data points in the train set: 5563, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503505 -> initscore=0.014021
[LightGBM] [Info] Start training from score 0.014021


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2801, number of negative: 2762
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.396757 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2436
[LightGBM] [Info] Number of data points in the train set: 5563, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503505 -> initscore=0.014021
[LightGBM] [Info] Start training from score 0.014021


...

[LightGBM] [Info] Number of positive: 2802, number of negative: 2762
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.129805 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2438
[LightGBM] [Info] Number of data points in the train set: 5564, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014378
[LightGBM] [Info] Start training from score 0.014378


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000246 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2439
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
Model: LGBM
Training time (mins): 7.93
Best parameters: {'learning_rate': 0.05, 'n_estimators': 100}
Validation accuracy: 0.8004600345025877



/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model: CatBoost
Training time (mins): 0.07
Best parameters: {'depth': 6, 'learning_rate': 0.1, 'n_estimators': 100}
Validation accuracy: 0.7952846463484762

Model: NaiveBayes
Training time (mins): 0.0
Best parameters: {'var_smoothing': 1e-08}
Validation accuracy: 0.6716503737780334



In [4]:
# Show results
valid_scores

,Classifier,Validation Accuracy,Training Time
0,LogisticRegression,0.770558,0.02
1,KNN,0.768258,0.01
2,SVC,0.793560,0.35
3,RandomForest,0.784359,0.04
4,LGBM,0.800460,7.93
5,CatBoost,0.795285,0.07
6,NaiveBayes,0.671650,0.00


In [5]:
# Show best parameters from grid search
clf_best_params

{'LogisticRegression': {'C': 0.5,
  'max_iter': 100,
  'penalty': 'l2',
  'solver': 'lbfgs'},
 'KNN': {'n_neighbors': 7, 'p': 1},
 'SVC': {'C': 1.5, 'gamma': 'scale', 'kernel': 'rbf'},
 'RandomForest': {'max_depth': 8, 'n_estimators': 150},
 'LGBM': {'learning_rate': 0.05, 'n_estimators': 100},
 'CatBoost': {'depth': 6, 'learning_rate': 0.1, 'n_estimators': 100},
 'NaiveBayes': {'var_smoothing': 1e-08}}

In [6]:
# lgbm_clf 和 catboost_clf 是之前通过 GridSearchCV 训练得到的最佳模型

# 获取最佳参数
lgbm_best_params = clf_best_params["LGBM"]
catboost_best_params = clf_best_params["CatBoost"]

# 使用最佳参数重新实例化模型
best_classifiers = {
    "LGBM": LGBMClassifier(**lgbm_best_params, random_state=0),
    "CatBoost": CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0),
}

# 训练模型
for key, classifier in best_classifiers.items():
    classifier.fit(X_train, y_train)
    print(f"{key} 训练完成")

import joblib

joblib.dump(best_classifiers["LGBM"], 'lgbm_best_model.joblib')
joblib.dump(best_classifiers["CatBoost"], 'catboost_best_model.joblib')




[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000306 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2439
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
LGBM 训练完成
CatBoost 训练完成


['catboost_best_model.joblib']

In [7]:
import joblib
import pandas as pd
import numpy as np

X = np.load("X.npy")
X_test = np.load("X_test.npy")

# 2. 从原始数据提取标签（如果未保存 y.npy）
# 如果已保存 y.npy，直接加载：
y = np.load("y.npy")
# 加载模型
lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# 使用加载的模型进行预测
lgbm_pred = lgbm_loaded_model.predict(X_test)
catboost_pred = catboost_loaded_model.predict(X_test)

# 加权投票法
ensemble_pred = (lgbm_pred * 0.6 + catboost_pred * 0.4) > 0.5
ensemble_pred = ensemble_pred.astype(bool)

test_data = pd.read_csv('test.csv')
# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
    'Transported': ensemble_pred
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
